In [26]:
import csv
import os
import re
import sys
import time
from pathlib import Path

import requests

os.environ["API_FOOTBALL_KEY"] = "cd44193e930c85a8a50c6219ab226905"
API_KEY = os.environ.get("API_FOOTBALL_KEY")
BASE_URL = "https://v3.football.api-sports.io"

LEAGUES = {
    39: "Premier League",
    140: "La Liga",
    135: "Serie A",
    78: "Bundesliga",
    61: "Ligue 1",
    2: "UEFA Champions League",
    3: "UEFA Europa League",
    4: "UEFA Conference League",
    180: "Championship",
    88: "Eredivisie (Nederlands)",
    94: "Primeira Liga (Portugal)",
}

SEASONS = [2022, 2023, 2024]

OUTPUT_DIR = Path("output")

RUN = {
    "teams": True,
    "fixtures": True,
    "standings": True,
    "team_statistics": True,
    "injuries": False,
    "sidelined": False,
    "missing_players": False,
}

REQUESTS_PER_MINUTE_LIMIT = 10
MIN_SECONDS_BETWEEN_REQUESTS = 60 / REQUESTS_PER_MINUTE_LIMIT + 0.5

_last_request_time = 0.0


class DailyQuotaExceeded(Exception):
    pass


def _headers():
    if not API_KEY:
        sys.exit("Hiányzik az API_FOOTBALL_KEY environmental variable")
    return {"x-apisports-key": API_KEY}


def _throttle():
    global _last_request_time
    elapsed = time.time() - _last_request_time
    wait = MIN_SECONDS_BETWEEN_REQUESTS - elapsed
    if wait > 0:
        print(f"  Waiting {wait:.1f}s because of the minute limit...")
        time.sleep(wait)


def api_get(endpoint: str, params: dict) -> dict:
    global _last_request_time
    url = f"{BASE_URL}/{endpoint}"

    for _ in range(10):
        _throttle()
        response = requests.get(url, headers=_headers(), params=params, timeout=30)
        _last_request_time = time.time()

        remaining_day = response.headers.get("x-ratelimit-requests-remaining")
        remaining_minute = response.headers.get("X-RateLimit-Remaining")
        if remaining_day is not None:
            print(f"  [{endpoint}] daily quota left: {remaining_day}")
        if remaining_minute is not None:
            print(f"  [{endpoint}] quota left in this minute: {remaining_minute}")

        if response.status_code == 429:
            print("  Error 429, waiting 60s...")
            time.sleep(60)
            continue

        response.raise_for_status()
        payload = response.json()

        errors = payload.get("errors")
        if errors:
            print(f"  API hiba ({endpoint}): {errors}")
            error_text = str(errors).lower()
            if "limit for the day" in error_text or "daily" in error_text:
                raise DailyQuotaExceeded(errors)

        return payload

    raise RuntimeError(f"Unsuccessful query {endpoint} (after too much error 429)")


def api_get_all_pages(endpoint: str, params: dict) -> list:
    results = []
    page = 1
    while True:
        payload = api_get(endpoint, {**params, "page": page})
        results.extend(payload.get("response", []))

        total_pages = payload.get("paging", {}).get("total", 1)
        if page >= total_pages:
            break
        page += 1

    return results


def write_csv(filename: str, rows: list[dict], fieldnames: list[str]):
    if not rows:
        print(f"  {filename}: no data: skipped")
        return

    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / filename
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Saved: {path} ({len(rows)} line(s))")


def _to_int(value):
    if value is None or value == "":
        return None
    try:
        return int(value)
    except (ValueError, TypeError):
        return value


def load_existing_rows(filename: str) -> list[dict]:
    path = OUTPUT_DIR / filename
    if not path.exists():
        return []

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []
        rows = list(reader)

    if rows and "league_id" not in fieldnames:
        MIGRATE_LEAGUE_ID = 39
        MIGRATE_SEASON = 2024
        print(f"  [migrating] {filename}: old format, {len(rows)} line(s) -> "
              f"league_id={MIGRATE_LEAGUE_ID} ({LEAGUES.get(MIGRATE_LEAGUE_ID)}), "
              f"season={MIGRATE_SEASON}.")
        for row in rows:
            row["league_id"] = MIGRATE_LEAGUE_ID
            row["league_name"] = LEAGUES.get(MIGRATE_LEAGUE_ID)
            row["season"] = MIGRATE_SEASON

    key_fields = ("league_id", "season", "team_id", "player_id", "fixture_id", "round",
                  "home_goals", "away_goals")
    for row in rows:
        for field in key_fields:
            if field in row:
                row[field] = _to_int(row[field])

    return rows


def fetch_teams(league_id: int, season: int) -> list[dict]:
    print(f"Csapatok lekérése (league={league_id}, season={season})...")
    payload = api_get("teams", {"league": league_id, "season": season})
    rows = []
    for item in payload.get("response", []):
        team = item["team"]
        venue = item.get("venue", {})
        rows.append({
            "league_id": league_id,
            "league_name": LEAGUES.get(league_id),
            "season": season,
            "team_id": team["id"],
            "name": team["name"],
            "code": team.get("code"),
            "founded": team.get("founded"),
            "venue_name": venue.get("name"),
            "venue_city": venue.get("city"),
        })
    return rows


def fetch_fixtures(league_id: int, season: int) -> list[dict]:
    print(f"Querying fixture list (league={league_id}, season={season})...")
    payload = api_get("fixtures", {"league": league_id, "season": season})
    rows = []
    for item in payload.get("response", []):
        fixture = item["fixture"]
        teams = item["teams"]
        goals = item["goals"]
        rows.append({
            "league_id": league_id,
            "league_name": LEAGUES.get(league_id),
            "season": season,
            "fixture_id": fixture["id"],
            "date": fixture["date"],
            "status": fixture["status"]["short"],
            "referee": fixture.get("referee"),
            "round": item["league"].get("round"),
            "home_team_id": teams["home"]["id"],
            "home_team_name": teams["home"]["name"],
            "away_team_id": teams["away"]["id"],
            "away_team_name": teams["away"]["name"],
            "home_goals": goals["home"],
            "away_goals": goals["away"],
        })
    return rows


def _round_sort_key(round_label) -> int:
    if isinstance(round_label, int):
        return round_label
    match = re.search(r"(\d+)", round_label or "")
    return int(match.group(1)) if match else 0


def compute_standings_by_round(fixtures: list[dict]) -> list[dict]:
    print("Calculating standings each round from fixtures data...")

    rows = []
    valid_fixtures = [f for f in fixtures if f.get("league_id") is not None and f.get("season") is not None]
    skipped = len(fixtures) - len(valid_fixtures)
    if skipped:
        print(f"  {skipped} fixture line skipped (probably no league_id/season "
              f"came from non-migrated old CSV).")

    keys = sorted({(f["league_id"], f["season"]) for f in valid_fixtures})
    for league_id, season in keys:
        subset = [f for f in valid_fixtures if f["league_id"] == league_id and f["season"] == season]

        teams = {}
        for f in subset:
            teams[f["home_team_id"]] = f["home_team_name"]
            teams[f["away_team_id"]] = f["away_team_name"]

        stats = {
            team_id: {"played": 0, "wins": 0, "draws": 0, "losses": 0,
                      "goals_for": 0, "goals_against": 0, "points": 0}
            for team_id in teams
        }

        rounds = sorted({f["round"] for f in subset if f["round"]}, key=_round_sort_key)

        for round_label in rounds:
            round_matches = [
                f for f in subset
                if f["round"] == round_label and f["status"] == "FT" and f["home_goals"] is not None
            ]
            if not round_matches:
                continue

            for match in round_matches:
                home_id, away_id = match["home_team_id"], match["away_team_id"]
                home_goals, away_goals = match["home_goals"], match["away_goals"]

                stats[home_id]["played"] += 1
                stats[away_id]["played"] += 1
                stats[home_id]["goals_for"] += home_goals
                stats[home_id]["goals_against"] += away_goals
                stats[away_id]["goals_for"] += away_goals
                stats[away_id]["goals_against"] += home_goals

                if home_goals > away_goals:
                    stats[home_id]["wins"] += 1
                    stats[home_id]["points"] += 3
                    stats[away_id]["losses"] += 1
                elif home_goals < away_goals:
                    stats[away_id]["wins"] += 1
                    stats[away_id]["points"] += 3
                    stats[home_id]["losses"] += 1
                else:
                    stats[home_id]["draws"] += 1
                    stats[away_id]["draws"] += 1
                    stats[home_id]["points"] += 1
                    stats[away_id]["points"] += 1

            snapshot = sorted(
                stats.items(),
                key=lambda item: (
                    -item[1]["points"],
                    -(item[1]["goals_for"] - item[1]["goals_against"]),
                    -item[1]["goals_for"],
                ),
            )

            for rank, (team_id, s) in enumerate(snapshot, start=1):
                rows.append({
                    "league_id": league_id,
                    "league_name": LEAGUES.get(league_id),
                    "season": season,
                    "round": round_label,
                    "rank": rank,
                    "team_id": team_id,
                    "team_name": teams[team_id],
                    "points": s["points"],
                    "played": s["played"],
                    "wins": s["wins"],
                    "draws": s["draws"],
                    "losses": s["losses"],
                    "goals_for": s["goals_for"],
                    "goals_against": s["goals_against"],
                    "goal_difference": s["goals_for"] - s["goals_against"],
                })

    return rows


def fetch_team_statistics(league_id: int, season: int, teams: list[dict]) -> list[dict]:
    print(f"Querying team statistics (league={league_id}, season={season})...")
    rows = []
    for team in teams:
        team_id = team["team_id"]
        print(f" -> {team['name']}")
        payload = api_get(
            "teams/statistics",
            {"league": league_id, "season": season, "team": team_id},
        )
        stats = payload.get("response", {})
        if not stats:
            continue

        fixtures = stats.get("fixtures", {})
        goals = stats.get("goals", {})
        cards = stats.get("cards", {})

        played_total = fixtures.get("played", {}).get("total", 0) or 1

        yellow_total = sum((v.get("total") or 0) for v in cards.get("yellow", {}).values())
        red_total = sum((v.get("total") or 0) for v in cards.get("red", {}).values())

        rows.append({
            "league_id": league_id,
            "league_name": LEAGUES.get(league_id),
            "season": season,
            "team_id": team_id,
            "team_name": team["name"],
            "played": fixtures.get("played", {}).get("total"),
            "goals_for_total": goals.get("for", {}).get("total", {}).get("total"),
            "goals_for_avg": goals.get("for", {}).get("average", {}).get("total"),
            "goals_against_total": goals.get("against", {}).get("total", {}).get("total"),
            "goals_against_avg": goals.get("against", {}).get("average", {}).get("total"),
            "goals_for_avg_home": goals.get("for", {}).get("average", {}).get("home"),
            "goals_for_avg_away": goals.get("for", {}).get("average", {}).get("away"),
            "goals_against_avg_home": goals.get("against", {}).get("average", {}).get("home"),
            "goals_against_avg_away": goals.get("against", {}).get("average", {}).get("away"),
            "form": stats.get("form"),
            "clean_sheets_total": stats.get("clean_sheet", {}).get("total"),
            "failed_to_score_total": stats.get("failed_to_score", {}).get("total"),
            "yellow_cards_total": yellow_total,
            "red_cards_total": red_total,
            "yellow_cards_per_match": round(yellow_total / played_total, 2),
        })

    return rows


def fetch_injuries(league_id: int, season: int) -> list[dict]:
    print(f"Querying injuries/bans (league={league_id}, season={season})...")
    payload = api_get("injuries", {"league": league_id, "season": season})
    rows = []
    for item in payload.get("response", []):
        player = item["player"]
        team = item["team"]
        fixture = item.get("fixture", {})
        rows.append({
            "league_id": league_id,
            "league_name": LEAGUES.get(league_id),
            "season": season,
            "team_id": team["id"],
            "team_name": team["name"],
            "player_id": player["id"],
            "player_name": player["name"],
            "type": player.get("type"),
            "reason": player.get("reason"),
            "fixture_id": fixture.get("id"),
            "fixture_date": fixture.get("date"),
        })
    return rows


def fetch_sidelined(league_id: int, season: int, injuries: list[dict],
                     already_fetched_players: set) -> list[dict]:
    relevant = [row for row in injuries if row["league_id"] == league_id and row["season"] == season]
    player_ids = sorted({row["player_id"] for row in relevant})
    if not player_ids:
        print(f"No injured players (league={league_id}, season={season}): skipped sidelined")
        return []

    new_player_ids = [pid for pid in player_ids if pid not in already_fetched_players]
    skipped = len(player_ids) - len(new_player_ids)
    if skipped:
        print(f"  [cache] {skipped} players' sidelined data has already been created: skipped")
    if not new_player_ids:
        return []

    player_info = {
        row["player_id"]: (row["player_name"], row["team_id"], row["team_name"])
        for row in relevant
    }

    print(f"Querying injury/ban duration for {len(new_player_ids)} new player(s)...")
    rows = []
    for player_id in new_player_ids:
        player_name, team_id, team_name = player_info[player_id]
        print(f" -> {player_name}")
        payload = api_get("sidelined", {"player": player_id})

        for entry in payload.get("response", []):
            periods = entry.get("sidelined", [entry]) if isinstance(entry, dict) else [entry]
            for period in periods:
                rows.append({
                    "league_id": league_id,
                    "league_name": LEAGUES.get(league_id),
                    "season": season,
                    "player_id": player_id,
                    "player_name": player_name,
                    "team_id": team_id,
                    "team_name": team_name,
                    "type": period.get("type"),
                    "start": period.get("start"),
                    "end": period.get("end"),
                })

        already_fetched_players.add(player_id)

    return rows


def fetch_missing_players(league_id: int, season: int, injuries: list[dict],
                           existing_keys: set) -> list[dict]:
    relevant = [row for row in injuries if row["league_id"] == league_id and row["season"] == season]
    affected_team_ids = sorted({row["team_id"] for row in relevant})
    if not affected_team_ids:
        print(f"No teams involved (league={league_id}, season={season}): missing_players skipped")
        return []

    new_team_ids = [tid for tid in affected_team_ids if (league_id, season, tid) not in existing_keys]
    skipped = len(affected_team_ids) - len(new_team_ids)
    if skipped:
        print(f"  [cache] {skipped} team's missing_players data has already been created: skipped")
    if not new_team_ids:
        return []

    print(f"Missing players' statisctics ({len(new_team_ids)} teams involved)...")
    injured_player_ids = {row["player_id"] for row in relevant}

    rows = []
    for team_id in new_team_ids:
        print(f" -> team_id={team_id}")
        players = api_get_all_pages(
            "players", {"league": league_id, "season": season, "team": team_id}
        )
        for item in players:
            player = item["player"]
            if player["id"] not in injured_player_ids:
                continue

            stats_list = item.get("statistics", [])
            stats = stats_list[0] if stats_list else {}
            games = stats.get("games", {})
            goals = stats.get("goals", {})

            minutes = games.get("minutes") or 0
            goals_scored = goals.get("total") or 0
            minutes_per_goal = round(minutes / goals_scored, 1) if goals_scored else None

            rows.append({
                "league_id": league_id,
                "league_name": LEAGUES.get(league_id),
                "season": season,
                "team_id": team_id,
                "player_id": player["id"],
                "player_name": player["name"],
                "position": games.get("position"),
                "minutes_played": minutes,
                "goals_scored": goals_scored,
                "minutes_per_goal": minutes_per_goal,
            })

        existing_keys.add((league_id, season, team_id))

    return rows


def process_combo(league_id: int, season: int,
                   all_teams: list, all_fixtures: list, all_team_stats: list,
                   all_injuries: list, all_sidelined: list, all_missing_players: list,
                   existing_teams_keys: set, existing_fixtures_keys: set,
                   existing_team_stats_keys: set, existing_injuries_keys: set,
                   existing_missing_players_keys: set, existing_sidelined_players: set) -> None:
    print(f"\n=== {LEAGUES[league_id]} {season} ===")

    teams_for_combo = []
    need_team_list = RUN["teams"] or RUN["team_statistics"]
    if need_team_list:
        if (league_id, season) in existing_teams_keys:
            print("  [cache] teamlist has already been created")
            teams_for_combo = [r for r in all_teams
                                if r["league_id"] == league_id and r["season"] == season]
        else:
            teams_for_combo = fetch_teams(league_id, season)
            all_teams.extend(teams_for_combo)
            existing_teams_keys.add((league_id, season))

    fixtures_for_combo = []
    if RUN["fixtures"] or RUN["standings"]:
        if (league_id, season) in existing_fixtures_keys:
            print("  [cache] fixtures has already been created")
            fixtures_for_combo = [r for r in all_fixtures
                                   if r["league_id"] == league_id and r["season"] == season]
        else:
            fixtures_for_combo = fetch_fixtures(league_id, season)
            all_fixtures.extend(fixtures_for_combo)
            existing_fixtures_keys.add((league_id, season))

    if RUN["team_statistics"]:
        missing_teams = [t for t in teams_for_combo
                          if (league_id, season, t["team_id"]) not in existing_team_stats_keys]
        if not missing_teams:
            print("  [cache] all teams' statisctics has already been created")
        else:
            new_stats = fetch_team_statistics(league_id, season, missing_teams)
            all_team_stats.extend(new_stats)
            existing_team_stats_keys.update(
                (league_id, season, t["team_id"]) for t in missing_teams
            )

    injuries_for_combo = []
    if RUN["injuries"]:
        if (league_id, season) in existing_injuries_keys:
            print("  [cache] injuries has already been created")
            injuries_for_combo = [r for r in all_injuries
                                   if r["league_id"] == league_id and r["season"] == season]
        else:
            injuries_for_combo = fetch_injuries(league_id, season)
            all_injuries.extend(injuries_for_combo)
            existing_injuries_keys.add((league_id, season))

    if RUN["sidelined"]:
        if not RUN["injuries"]:
            print("Querrying sidelined needs injuries section: skipped")
        else:
            new_sidelined = fetch_sidelined(
                league_id, season, injuries_for_combo, existing_sidelined_players
            )
            all_sidelined.extend(new_sidelined)

    if RUN["missing_players"]:
        if not RUN["injuries"]:
            print("Querrying missing_players needs injuries section: skipped")
        else:
            new_missing = fetch_missing_players(
                league_id, season, injuries_for_combo, existing_missing_players_keys
            )
            all_missing_players.extend(new_missing)


def main():
    all_teams = load_existing_rows("teams.csv")
    all_fixtures = load_existing_rows("fixtures.csv")
    all_team_stats = load_existing_rows("team_statistics.csv")
    all_injuries = load_existing_rows("injuries.csv")
    all_sidelined = load_existing_rows("sidelined.csv")
    all_missing_players = load_existing_rows("missing_players.csv")

    existing_teams_keys = {(r["league_id"], r["season"]) for r in all_teams
                            if r.get("league_id") is not None and r.get("season") is not None}
    existing_fixtures_keys = {(r["league_id"], r["season"]) for r in all_fixtures
                               if r.get("league_id") is not None and r.get("season") is not None}
    existing_team_stats_keys = {(r["league_id"], r["season"], r["team_id"]) for r in all_team_stats
                                 if r.get("league_id") is not None and r.get("season") is not None
                                 and r.get("team_id") is not None}
    existing_injuries_keys = {(r["league_id"], r["season"]) for r in all_injuries
                               if r.get("league_id") is not None and r.get("season") is not None}
    existing_missing_players_keys = {(r["league_id"], r["season"], r["team_id"]) for r in all_missing_players
                                      if r.get("league_id") is not None and r.get("season") is not None
                                      and r.get("team_id") is not None}
    existing_sidelined_players = {r["player_id"] for r in all_sidelined if r.get("player_id") is not None}

    print(f"Cache: {len(existing_teams_keys)} league/season teams, "
          f"{len(existing_fixtures_keys)} league/season fixtures, "
          f"{len(existing_team_stats_keys)} team-statistic, "
          f"{len(existing_injuries_keys)} league/season injuries, "
          f"{len(existing_sidelined_players)} sidelined játékos, "
          f"{len(existing_missing_players_keys)} missing_players team/season")

    combos = [(league_id, season) for league_id in LEAGUES for season in SEASONS]
    print(f"{len(combos)} league/season combination in total: "
          f"{[(LEAGUES[l], s) for l, s in combos]}")

    for league_id, season in combos:
        try:
            process_combo(
                league_id, season,
                all_teams, all_fixtures, all_team_stats, all_injuries,
                all_sidelined, all_missing_players,
                existing_teams_keys, existing_fixtures_keys, existing_team_stats_keys,
                existing_injuries_keys, existing_missing_players_keys, existing_sidelined_players,
            )
        except DailyQuotaExceeded as e:
            print(f"\nDaily API quota is over ({e}).")
            print("Script is stopping, saving the data that it successfully got")
            break

    if RUN["standings"] and all_fixtures:
        standings_rows = compute_standings_by_round(all_fixtures)
        write_csv(
            "standings.csv",
            standings_rows,
            ["league_id", "league_name", "season", "round", "rank", "team_id", "team_name",
             "points", "played", "wins", "draws", "losses",
             "goals_for", "goals_against", "goal_difference"],
        )

    write_csv(
        "teams.csv", all_teams,
        ["league_id", "league_name", "season", "team_id", "name", "code",
         "founded", "venue_name", "venue_city"],
    )

    write_csv(
        "fixtures.csv", all_fixtures,
        ["league_id", "league_name", "season", "fixture_id", "date", "status", "referee", "round",
         "home_team_id", "home_team_name", "away_team_id", "away_team_name",
         "home_goals", "away_goals"],
    )

    if RUN["team_statistics"]:
        write_csv(
            "team_statistics.csv", all_team_stats,
            ["league_id", "league_name", "season", "team_id", "team_name", "played",
             "goals_for_total", "goals_for_avg", "goals_against_total", "goals_against_avg",
             "goals_for_avg_home", "goals_for_avg_away",
             "goals_against_avg_home", "goals_against_avg_away",
             "form", "clean_sheets_total", "failed_to_score_total",
             "yellow_cards_total", "red_cards_total", "yellow_cards_per_match"],
        )

    if RUN["injuries"]:
        write_csv(
            "injuries.csv", all_injuries,
            ["league_id", "league_name", "season", "team_id", "team_name", "player_id", "player_name",
             "type", "reason", "fixture_id", "fixture_date"],
        )

    if RUN["sidelined"]:
        write_csv(
            "sidelined.csv", all_sidelined,
            ["league_id", "league_name", "season", "player_id", "player_name", "team_id",
             "team_name", "type", "start", "end"],
        )

    if RUN["missing_players"]:
        write_csv(
            "missing_players.csv", all_missing_players,
            ["league_id", "league_name", "season", "team_id", "player_id", "player_name",
             "position", "minutes_played", "goals_scored", "minutes_per_goal"],
        )

    print("\nDone. CSV files are in the '/output' folder.")


if __name__ == "__main__":
    main()

  [migrating] injuries.csv: old format, 3168 line(s) -> league_id=39 (Premier League), season=2024.
  [migrating] missing_players.csv: old format, 440 line(s) -> league_id=39 (Premier League), season=2024.
Cache: 9 league/season teams, 31 league/season fixtures, 180 team-statistic, 1 league/season injuries, 0 sidelined játékos, 20 missing_players team/season
33 league/season combination in total: [('Premier League', 2022), ('Premier League', 2023), ('Premier League', 2024), ('La Liga', 2022), ('La Liga', 2023), ('La Liga', 2024), ('Serie A', 2022), ('Serie A', 2023), ('Serie A', 2024), ('Bundesliga', 2022), ('Bundesliga', 2023), ('Bundesliga', 2024), ('Ligue 1', 2022), ('Ligue 1', 2023), ('Ligue 1', 2024), ('UEFA Champions League', 2022), ('UEFA Champions League', 2023), ('UEFA Champions League', 2024), ('UEFA Europa League', 2022), ('UEFA Europa League', 2023), ('UEFA Europa League', 2024), ('UEFA Conference League', 2022), ('UEFA Conference League', 2023), ('UEFA Conference League', 

In [27]:
import pandas as pd


def get_fixture_info(fixtures: pd.DataFrame, fixture_id: int) -> dict:
    row = fixtures.loc[fixtures["fixture_id"] == fixture_id]

    if row.empty:
        raise ValueError(f"Fixture_id not found: {fixture_id}")

    row = row.iloc[0]
    return {
        "team1": row["home_team_id"],
        "team2": row["away_team_id"],
        "team1_goals": row["home_goals"],
        "team2_goals": row["away_goals"],
        "round": str(row["round"]),
        "league_id": row["league_id"],
        "season": row["season"],
        "date": pd.to_datetime(row["date"]),
    }


def get_attack_defense_stats(statistics: pd.DataFrame, team_id: int,
                              league_id: int, season: int) -> dict:
    row = statistics.loc[
        (statistics["team_id"] == team_id)
        & (statistics["league_id"] == league_id)
        & (statistics["season"] == season)
    ].iloc[0]
    return {
        "attack": row["goals_for_avg"],
        "defense": row["goals_against_avg"],
    }


def compute_league_avg_goals(statistics: pd.DataFrame) -> pd.Series:
    return statistics.groupby(["league_id", "season"])["goals_for_avg"].mean()


def expected_goals(team_attack: float, opp_defense: float, league_avg) -> float | None:
    if league_avg is None or pd.isna(league_avg) or league_avg <= 0:
        return None
    exp = (team_attack * opp_defense) / league_avg
    return max(exp, 0.05)


def update_ewma(previous: float | None, new_ratio: float, alpha: float = 0.35) -> float:
    if previous is None or pd.isna(previous):
        return new_ratio
    return alpha * new_ratio + (1 - alpha) * previous


def build_dataset(fixtures: pd.DataFrame, statistics: pd.DataFrame, alpha: float = 0.35) -> pd.DataFrame:
    played = fixtures.dropna(subset=["home_goals", "away_goals"]).copy()
    played["date"] = pd.to_datetime(played["date"])
    played = played.sort_values("date").reset_index(drop=True)

    league_avg_goals = compute_league_avg_goals(statistics)

    ewma_attack: dict[int, float] = {}
    ewma_defense: dict[int, float] = {}
    last_match_date: dict[int, pd.Timestamp] = {}

    NEUTRAL_FORM = 1.0
    NEUTRAL_DAYS_SINCE = 365

    rows = []

    for _, fx in played.iterrows():
        league_id = fx["league_id"]
        season = fx["season"]
        home_id = fx["home_team_id"]
        away_id = fx["away_team_id"]
        match_date = fx["date"]

        try:
            home_stats = get_attack_defense_stats(statistics, home_id, league_id, season)
            away_stats = get_attack_defense_stats(statistics, away_id, league_id, season)
        except (IndexError, ValueError):
            continue

        league_avg = league_avg_goals.get((league_id, season))

        home_attack_form = ewma_attack.get(home_id, NEUTRAL_FORM)
        home_defense_form = ewma_defense.get(home_id, NEUTRAL_FORM)
        away_attack_form = ewma_attack.get(away_id, NEUTRAL_FORM)
        away_defense_form = ewma_defense.get(away_id, NEUTRAL_FORM)

        home_days_since = (
            (match_date - last_match_date[home_id]).total_seconds() / 86400
            if home_id in last_match_date else NEUTRAL_DAYS_SINCE
        )
        away_days_since = (
            (match_date - last_match_date[away_id]).total_seconds() / 86400
            if away_id in last_match_date else NEUTRAL_DAYS_SINCE
        )

        rows.append({
            "fixture_id": fx["fixture_id"],
            "league_id": league_id,
            "season": season,
            "round": str(fx["round"]),
            "home_attack": home_stats["attack"],
            "home_defense": home_stats["defense"],
            "away_attack": away_stats["attack"],
            "away_defense": away_stats["defense"],
            "home_goals_scored_lastN": home_attack_form,
            "home_goals_conceded_lastN": home_defense_form,
            "away_goals_scored_lastN": away_attack_form,
            "away_goals_conceded_lastN": away_defense_form,
            "home_days_since_last_match": home_days_since,
            "away_days_since_last_match": away_days_since,
            "home_goals": fx["home_goals"],
            "away_goals": fx["away_goals"],
        })

        exp_home_goals = expected_goals(home_stats["attack"], away_stats["defense"], league_avg)
        exp_away_goals = expected_goals(away_stats["attack"], home_stats["defense"], league_avg)

        if exp_home_goals is not None:
            ratio = fx["home_goals"] / exp_home_goals
            ewma_attack[home_id] = update_ewma(ewma_attack.get(home_id), ratio, alpha)
            ewma_defense[away_id] = update_ewma(ewma_defense.get(away_id), ratio, alpha)

        if exp_away_goals is not None:
            ratio = fx["away_goals"] / exp_away_goals
            ewma_attack[away_id] = update_ewma(ewma_attack.get(away_id), ratio, alpha)
            ewma_defense[home_id] = update_ewma(ewma_defense.get(home_id), ratio, alpha)

        last_match_date[home_id] = match_date
        last_match_date[away_id] = match_date

    team_form_state = {
        "ewma_attack": ewma_attack,
        "ewma_defense": ewma_defense,
        "last_match_date": last_match_date,
    }

    return pd.DataFrame(rows), team_form_state


def main():
    import json
    import os

    fixtures = pd.read_csv("output/fixtures.csv")
    statistics = pd.read_csv("output/team_statistics.csv")

    dataset, team_form_state = build_dataset(fixtures, statistics, alpha=0.35)
    print(f"Dataset size: {dataset.shape}")
    print(dataset.head())
    dataset.to_csv("output/dataset.csv", index=False)
    print("\Saved: output/dataset.csv")

    os.makedirs("output", exist_ok=True)
    serializable_state = {
        "ewma_attack": team_form_state["ewma_attack"],
        "ewma_defense": team_form_state["ewma_defense"],
        "last_match_date": {
            team_id: str(date) for team_id, date in team_form_state["last_match_date"].items()
        },
    }
    with open("output/team_form_state.json", "w") as f:
        json.dump(serializable_state, f, indent=2)
    print("Saved: output/team_form_state.json")


if __name__ == "__main__":
    main()

<>:153: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:153: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\koosb\AppData\Local\Temp\ipykernel_13512\1801439583.py:153: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  print("\Saved: output/dataset.csv")


Dataset size: (4725, 16)
   fixture_id  league_id  season               round  home_attack  \
0      871164         78    2022  Regular Season - 1          1.7   
1      867946         39    2022                   1          1.1   
2      871474         61    2022  Regular Season - 1          1.7   
3      867947         39    2022                   1          1.4   
4      871168         78    2022  Regular Season - 1          1.5   

   home_defense  away_attack  away_defense  home_goals_scored_lastN  \
0           1.5          2.7           1.1                      1.0   
1           1.3          2.3           1.1                      1.0   
2           1.2          0.6           1.9                      1.0   
3           1.4          2.0           1.2                      1.0   
4           1.6          1.4           1.7                      1.0   

   home_goals_conceded_lastN  away_goals_scored_lastN  \
0                        1.0                      1.0   
1                  